# TP2 Deep Learning : Partie 2 : Classification MNIST avec un Vision Transformer (ViT)

**Objectif :** construire un Vision Transformer (ViT) **from scratch** avec PyTorch pour reconnaître des chiffres manuscrits (0 à 9), en utilisant le GPU, puis comparer ses performances avec celles des modèles de la Partie 1.


In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hojjatk/mnist-dataset/t10k-images.idx3-ubyte
/kaggle/input/datasets/hojjatk/mnist-dataset/t10k-labels.idx1-ubyte
/kaggle/input/datasets/hojjatk/mnist-dataset/train-images.idx3-ubyte
/kaggle/input/datasets/hojjatk/mnist-dataset/train-labels.idx1-ubyte
/kaggle/input/datasets/hojjatk/mnist-dataset/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte
/kaggle/input/datasets/hojjatk/mnist-dataset/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte
/kaggle/input/datasets/hojjatk/mnist-dataset/train-images-idx3-ubyte/train-images-idx3-ubyte
/kaggle/input/datasets/hojjatk/mnist-dataset/train-labels-idx1-ubyte/train-labels-idx1-ubyte


In [11]:
import numpy as np

from tqdm import tqdm, trange

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

from torchvision.transforms import ToTensor
from torchvision.datasets.mnist import MNIST

np.random.seed(0)
torch.manual_seed(0)

In [12]:
def main():
  
    # 1 Chargement des données
    # On utilise ToTensor() pour convertir les images en tenseurs (pixels entre 0 et 1)
    transform = ToTensor()
    
    # On charge MNIST 
    train_set = MNIST(root='./../datasets', train=True, download=True, transform=transform)
    test_set = MNIST(root='./../datasets', train=False, download=True, transform=transform)
    
    # On crée les DataLoaders (batch_size=128 )
    train_loader = DataLoader(train_set, shuffle=True, batch_size=128)
    test_loader = DataLoader(test_set, shuffle=False, batch_size=128)

   
    # 2 Définition du modèle et des options d'entraînement
  
    # On choisit le device (GPU si dispo, sinon CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device: ", device, f"({torch.cuda.get_device_name(device)})" if torch.cuda.is_available() else "")
    
    #  modèle ViT 
    # Paramètres : 
    # - (1, 28, 28) : taille d'une image (1 canal, 28x28)
    # - n_patches=7 : on découpe l'image en 7x7 = 49 patchs
    # - n_blocks=2 : 2 blocs Transformer
    # - hidden_d=8 : 8 dimension cachée 
    # - n_heads=2 : 2 têtes d'attention
    # - out_d=10 : 10 classes (chiffres 0 à 9)
    model = MyViT((1, 28, 28), n_patches=7, n_blocks=2, hidden_d=8, n_heads=2, out_d=10).to(device)
    
    N_EPOCHS = 5  # Nombre d'epochs
    LR = 0.005    # Learning rate

    
    # 3 Boucle d'entraînement
    optimizer = Adam(model.parameters(), lr=LR)
    criterion = CrossEntropyLoss()
    
    # 
    for epoch in trange(N_EPOCHS, desc="Training"):
        train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1} in training", leave=False):
            x, y = batch
            x, y = x.to(device), y.to(device)
            
            # Forward pass : le modele prédit les 10 scores
            y_hat = model(x)
            loss = criterion(y_hat, y) # Loss
            
            # Accumulation de la loss pour affichage
            train_loss += loss.detach().cpu().item() / len(train_loader)
            
            # Backward pass + mise à jour des poids
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        print(f"Epoch {epoch + 1}/{N_EPOCHS} loss: {train_loss:.2f}")


    #  test

    with torch.no_grad():
        correct, total = 0, 0
        test_loss = 0.0
        for batch in tqdm(test_loader, desc="Testing"):
            x, y = batch
            x, y = x.to(device), y.to(device)
            
            y_hat = model(x)
            loss = criterion(y_hat, y)
            test_loss += loss.detach().cpu().item() / len(test_loader)
            
            # Comptage des bonnes prédictions
            correct += torch.sum(torch.argmax(y_hat, dim=1) == y).detach().cpu().item()
            total += len(x)
        
        print(f"Test loss: {test_loss:.2f}")
        print(f"Test accuracy: {correct / total * 100:.2f}%")

In [13]:
# On commence avec ne classe vide qui hérite de nn.Module (le squelette PyTorch pour tout modèle)
class MyViT(nn.Module):

    # Le constructeur 
    def __init__(self):
        super(MyViT, self).__init__()

    def forward(self, images):
        pass

In [14]:
# On cree une fonction qui découpe l'image en petits carrés  ( patches)
def patchify(images, n_patches):

    
    # n = nombre d'images, c = nombre de canaux (1 pour le gris), h = hauteur, w = largeur
    n, c, h, w = images.shape

     # On vérifie que l'image est carrée (h == w), sinon, cette implémentation ne marche pas.
    assert h == w, "Patchify method is implemented for square images only"

    # On cree un grand tableau vide pour stocker tous les patchs.
    # Sa forme sera est (n, n_patches^2, dimension d'un patch).
    # Exemple pour MNIST : (n, 49, 16) -> n images, 49 patchs, 16 valeurs par patch.
    patches = torch.zeros(n, n_patches ** 2, h * w * c // n_patches ** 2)
    patch_size = h // n_patches

    for idx, image in enumerate(images):
        for i in range(n_patches):
            for j in range(n_patches):
                patch = image[:, i * patch_size: (i + 1) * patch_size, j * patch_size: (j + 1) * patch_size]
                patches[idx, i * n_patches + j] = patch.flatten()
    return patches

In [15]:
# On met à jour la classe MyViT pour qu'elle fasse le découpage et la projection linéaire.
class MyViT(nn.Module):

      # Le constructeur prend maintenant la taille de l'image (chw) et le nombre de patchs.
    def __init__(self, chw=(1, 28, 28), n_patches=7):
        # Super constructor
        super(MyViT, self).__init__()

        #  # On stocke les attributs pour pouvoir les réutiliser dans forward.
        self.chw = chw # (C, H, W)
        self.n_patches = n_patches

        assert chw[1] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        assert chw[2] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        self.patch_size = (chw[1] / n_patches, chw[2] / n_patches)

        # 1) on crée la couche de projection linéaire (linear mapper)
        # Elle transforme chaque patch de "input_d" dimensions en "hidden_d" dimensions.
        self.input_d = int(chw[0] * self.patch_size[0] * self.patch_size[1])
        self.linear_mapper = nn.Linear(self.input_d, self.hidden_d)

    # La méthode forward fait maintenant deux choses : patchify puis linear mapper.
    def forward(self, images):
        patches = patchify(images, self.n_patches)
        tokens = self.linear_mapper(patches)
        return tokens

In [18]:
# On met à jour la classe pour ajouter le "class token".
class MyViT(nn.Module):
    
    def __init__(self, chw=(1, 28, 28), n_patches=7, hidden_d=8):
        super(MyViT, self).__init__()

        # Attributs de base.
        self.chw = chw
        self.n_patches = n_patches
        self.hidden_d = hidden_d   # <-- NOUVEAU : on stocke la dimension cachée

        assert chw[1] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        assert chw[2] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        self.patch_size = (chw[1] / n_patches, chw[2] / n_patches)

        # 1) Le linear mapper (comme avant).
        self.input_d = int(chw[0] * self.patch_size[0] * self.patch_size[1])
        self.linear_mapper = nn.Linear(self.input_d, self.hidden_d)

        # 2) Le class token (NOUVEAU).
        # C'est un vecteur de paramètres (qui seront appris) de dimension hidden_d.
        # Il sera ajouté au début de chaque séquence de tokens.
        self.class_token = nn.Parameter(torch.rand(1, self.hidden_d))

    def forward(self, images):
        
        # Étape 1 : patchify.
        patches = patchify(images, self.n_patches)
        
        # Étape 2 : linear mapper.
        tokens = self.linear_mapper(patches)

        # Étape 3 (NOUVEAU) : on ajoute le class token au début de chaque séquence.
        # On empile verticalement le class token et les tokens de l'image.
        tokens = torch.stack([torch.vstack((self.class_token, tokens[i])) for i in range(len(tokens))])
        
        # On retourne les tokens (50 maintenant : 1 CLS + 49 patchs).
        return tokens

In [19]:
# Cette fonction crée les "positional embeddings".
# Ce sont des valeurs ajoutées à chaque token pour lui dire où il se trouve dans l'image.
def get_positional_embeddings(sequence_length, d):
    
    # On crée un tableau rempli de 1, de taille (nombre de tokens, dimension).
    result = torch.ones(sequence_length, d)
    
    # On boucle sur chaque position (i) et chaque dimension (j).
    for i in range(sequence_length):
        for j in range(d):
            
            # Pour les dimensions paires (j % 2 == 0), on utilise sin.
            # Pour les dimensions impaires, on utilise cos.
            # C'est la formule classique des Transformers (Vaswani et al., 2017).
            result[i][j] = np.sin(i / (10000 ** (j / d))) if j % 2 == 0 else np.cos(i / (10000 ** ((j - 1) / d)))
    
    # On retourne la matrice des positions.
    return result

In [20]:
# On met à jour la classe pour ajouter le positional encoding.
class MyViT(nn.Module):
    
    def __init__(self, chw=(1, 28, 28), n_patches=7, hidden_d=8):
        super(MyViT, self).__init__()

        self.chw = chw
        self.n_patches = n_patches
        self.hidden_d = hidden_d

        assert chw[1] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        assert chw[2] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        self.patch_size = (chw[1] / n_patches, chw[2] / n_patches)

        # 1) Linear mapper.
        self.input_d = int(chw[0] * self.patch_size[0] * self.patch_size[1])
        self.linear_mapper = nn.Linear(self.input_d, self.hidden_d)

        # 2) Class token.
        self.class_token = nn.Parameter(torch.rand(1, self.hidden_d))

        # 3) Positional embedding (NOUVEAU).
        # On stocke la matrice des positions comme un paramètre du modèle.
        # requires_grad = False signifie qu'on ne va pas l'entraîner : il reste fixe.
        self.pos_embed = nn.Parameter(torch.tensor(get_positional_embeddings(self.n_patches ** 2 + 1, self.hidden_d)))
        self.pos_embed.requires_grad = False

    def forward(self, images):
        
        # Étape 1 : patchify.
        patches = patchify(images, self.n_patches)
        
        # Étape 2 : linear mapper.
        tokens = self.linear_mapper(patches)

        # Étape 3 : ajout du class token.
        tokens = torch.stack([torch.vstack((self.class_token, tokens[i])) for i in range(len(tokens))])

        # Étape 4 (NOUVEAU) : ajout du positional encoding.
        # On répète la matrice de positions pour chaque image du batch,
        # puis on l'ajoute aux tokens.
        pos_embed = self.pos_embed.repeat(len(tokens), 1, 1)
        out = tokens + pos_embed
        
        # On retourne les tokens enrichis.
        return out

In [21]:
# Cette classe implémente le mécanisme d'attention multi-têtes.
# C'est le cœur du Transformer : chaque token regarde les autres et décide à qui faire attention.
class MyMSA(nn.Module):
    
    # Le constructeur prend la dimension d et le nombre de têtes (n_heads).
    def __init__(self, d, n_heads=2):
        super(MyMSA, self).__init__()
        self.d = d
        self.n_heads = n_heads

        # On vérifie que d est divisible par n_heads.
        assert d % n_heads == 0, f"Can't divide dimension {d} into {n_heads} heads"

        # On calcule la dimension de chaque tête (d / n_heads).
        d_head = int(d / n_heads)
        
        # On crée des couches linéaires pour Q, K, V, une par tête.
        # nn.ModuleList est une liste de modules PyTorch.
        self.q_mappings = nn.ModuleList([nn.Linear(d_head, d_head) for _ in range(self.n_heads)])
        self.k_mappings = nn.ModuleList([nn.Linear(d_head, d_head) for _ in range(self.n_heads)])
        self.v_mappings = nn.ModuleList([nn.Linear(d_head, d_head) for _ in range(self.n_heads)])
        
        self.d_head = d_head
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, sequences):
        
        # On va stocker les résultats de toutes les têtes dans cette liste.
        result = []
        
        # On boucle sur chaque séquence (chaque image du batch).
        for sequence in sequences:
            
            # On va stocker les résultats des têtes pour cette séquence.
            seq_result = []
            
            # On boucle sur chaque tête.
            for head in range(self.n_heads):
                
                # On récupère les couches Q, K, V de cette tête.
                q_mapping = self.q_mappings[head]
                k_mapping = self.k_mappings[head]
                v_mapping = self.v_mappings[head]

                # On extrait la portion de la séquence correspondant à cette tête.
                seq = sequence[:, head * self.d_head: (head + 1) * self.d_head]
                
                # On calcule Q, K, V pour cette portion.
                q, k, v = q_mapping(seq), k_mapping(seq), v_mapping(seq)

                # On calcule l'attention :
                # - q @ k.T : similarité entre chaque token et les autres
                # - / sqrt(d_head) : normalisation pour stabiliser
                # - softmax : transforme en probabilités (poids)
                attention = self.softmax(q @ k.T / (self.d_head ** 0.5))
                
                # On applique les poids d'attention aux valeurs V.
                seq_result.append(attention @ v)
            
            # On concatène les résultats de toutes les têtes (hstack = concaténation horizontale).
            result.append(torch.hstack(seq_result))
        
        # On empile les résultats pour reformer un batch.
        return torch.cat([torch.unsqueeze(r, dim=0) for r in result])

In [22]:
# Ce bloc assemble l'attention et une connexion résiduelle.
# C'est une "brique" du Transformer.
class MyViTBlock(nn.Module):
    
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(MyViTBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        # 1) LayerNorm : on normalise les tokens avant l'attention.
        self.norm1 = nn.LayerNorm(hidden_d)
        
        # 2) L'attention multi-têtes.
        self.mhsa = MyMSA(hidden_d, n_heads)

    def forward(self, x):
        
        # On applique : LayerNorm -> MSA -> ajout du résidu (x + ...).
        out = x + self.mhsa(self.norm1(x))
        
        # On retourne les tokens enrichis.
        return out

In [24]:
# On améliore le bloc en ajoutant un MLP après l'attention.
class MyViTBlock(nn.Module):
    
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(MyViTBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        # 1) LayerNorm pour l'attention.
        self.norm1 = nn.LayerNorm(hidden_d)
        
        # 2) L'attention multi-têtes.
        self.mhsa = MyMSA(hidden_d, n_heads)
        
        # 3) NOUVEAU : LayerNorm pour le MLP.
        self.norm2 = nn.LayerNorm(hidden_d)
        
        # 4) NOUVEAU : le MLP (2 couches linéaires avec GELU entre).
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, mlp_ratio * hidden_d),
            nn.GELU(),
            nn.Linear(mlp_ratio * hidden_d, hidden_d)
        )

    def forward(self, x):
        
        # Étape 1 : LayerNorm -> MSA -> résidu.
        out = x + self.mhsa(self.norm1(x))
        
        # Étape 2 (NOUVEAU) : LayerNorm -> MLP -> résidu.
        out = out + self.mlp(self.norm2(out))
        
        return out

In [26]:
# On assemble tous les blocs dans le modèle complet.
class MyViT(nn.Module):
    
    def __init__(self, chw, n_patches=7, n_blocks=2, hidden_d=8, n_heads=2, out_d=10):
        super(MyViT, self).__init__()
        
        # Attributs.
        self.chw = chw
        self.n_patches = n_patches
        self.n_blocks = n_blocks
        self.n_heads = n_heads
        self.hidden_d = hidden_d
        
        # Vérifications.
        assert chw[1] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        assert chw[2] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        self.patch_size = (chw[1] / n_patches, chw[2] / n_patches)

        # 1) Linear mapper.
        self.input_d = int(chw[0] * self.patch_size[0] * self.patch_size[1])
        self.linear_mapper = nn.Linear(self.input_d, self.hidden_d)
        
        # 2) Class token.
        self.class_token = nn.Parameter(torch.rand(1, self.hidden_d))
        
        # 3) Positional embedding (NOUVEAU : register_buffer au lieu de nn.Parameter).
        # register_buffer permet de stocker un tenseur sans qu'il soit un paramètre entraînable.
        self.register_buffer('positional_embeddings', get_positional_embeddings(n_patches ** 2 + 1, hidden_d), persistent=False)
        
        # 4) Les N blocs Transformer (NOUVEAU).
        self.blocks = nn.ModuleList([MyViTBlock(hidden_d, n_heads) for _ in range(n_blocks)])

    def forward(self, images):
        
        # Étape 1 : patchify.
        n, c, h, w = images.shape
        patches = patchify(images, self.n_patches).to(self.positional_embeddings.device)
        
        # Étape 2 : linear mapper.
        tokens = self.linear_mapper(patches)
        
        # Étape 3 : ajout du class token (cat au lieu de stack+vstack, plus efficace).
        tokens = torch.cat((self.class_token.expand(n, 1, -1), tokens), dim=1)
        
        # Étape 4 : ajout du positional encoding.
        out = tokens + self.positional_embeddings.repeat(n, 1, 1)
        
        # Étape 5 (NOUVEAU) : on fait passer les tokens dans les N blocs Transformer.
        for block in self.blocks:
            out = block(out)
        
        # On retourne les tokens enrichis (50 tokens de 8 dimensions).
        return out

In [27]:
# On ajoute la tête de classification finale.
class MyViT(nn.Module):
    
    def __init__(self, chw, n_patches=7, n_blocks=2, hidden_d=8, n_heads=2, out_d=10):
        super(MyViT, self).__init__()
        
        # Attributs.
        self.chw = chw
        self.n_patches = n_patches
        self.n_blocks = n_blocks
        self.n_heads = n_heads
        self.hidden_d = hidden_d
        
        # Vérifications.
        assert chw[1] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        assert chw[2] % n_patches == 0, "Input shape not entirely divisible by number of patches"
        self.patch_size = (chw[1] / n_patches, chw[2] / n_patches)

        # 1) Linear mapper.
        self.input_d = int(chw[0] * self.patch_size[0] * self.patch_size[1])
        self.linear_mapper = nn.Linear(self.input_d, self.hidden_d)
        
        # 2) Class token.
        self.class_token = nn.Parameter(torch.rand(1, self.hidden_d))
        
        # 3) Positional embedding.
        self.register_buffer('positional_embeddings', get_positional_embeddings(n_patches ** 2 + 1, hidden_d), persistent=False)
        
        # 4) Blocs Transformer.
        self.blocks = nn.ModuleList([MyViTBlock(hidden_d, n_heads) for _ in range(n_blocks)])
        
        # 5) NOUVEAU : le MLP de classification final.
        # Il prend le class token (de dimension hidden_d) et sort out_d scores (10).
        # Le Softmax transforme les scores en probabilités.
        self.mlp = nn.Sequential(
            nn.Linear(self.hidden_d, out_d),
            nn.Softmax(dim=-1)
        )

    def forward(self, images):
        
        # Étape 1 : patchify.
        n, c, h, w = images.shape
        patches = patchify(images, self.n_patches).to(self.positional_embeddings.device)
        
        # Étape 2 : linear mapper.
        tokens = self.linear_mapper(patches)
        
        # Étape 3 : ajout du class token.
        tokens = torch.cat((self.class_token.expand(n, 1, -1), tokens), dim=1)
        
        # Étape 4 : ajout du positional encoding.
        out = tokens + self.positional_embeddings.repeat(n, 1, 1)
        
        # Étape 5 : blocs Transformer.
        for block in self.blocks:
            out = block(out)
        
        # Étape 6 (NOUVEAU) : on extrait UNIQUEMENT le class token (le premier token de la séquence).
        out = out[:, 0]
        
        # Étape 7 (NOUVEAU) : on le passe dans le MLP final pour obtenir 10 scores.
        return self.mlp(out)

In [28]:
def main():
    # 1 Chargement des données
    transform = ToTensor()
    
    train_set = MNIST(root='./data', train=True, download=True, transform=transform)
    test_set  = MNIST(root='./data', train=False, download=True, transform=transform)
    
    train_loader = DataLoader(train_set, shuffle=True, batch_size=128)
    test_loader = DataLoader(test_set, shuffle=False, batch_size=128)

    # 2 Définition du modèle et des options d'entraînement
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device: ", device, f"({torch.cuda.get_device_name(device)})" if torch.cuda.is_available() else "")
    
    # LA LIGNE QUI CHANGE : on utilise maintenant la version complète de MyViT
    model = MyViT((1, 28, 28), n_patches=7, n_blocks=2, hidden_d=8, n_heads=2, out_d=10).to(device)
    
    N_EPOCHS = 5
    LR = 0.005

    # 3 Boucle d'entraînement
    optimizer = Adam(model.parameters(), lr=LR)
    criterion = CrossEntropyLoss()
    
    for epoch in trange(N_EPOCHS, desc="Training"):
        train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1} in training", leave=False):
            x, y = batch
            x, y = x.to(device), y.to(device)
            
            y_hat = model(x)
            loss = criterion(y_hat, y)
            
            train_loss += loss.detach().cpu().item() / len(train_loader)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        print(f"Epoch {epoch + 1}/{N_EPOCHS} loss: {train_loss:.2f}")

    # 4 Test
    with torch.no_grad():
        correct, total = 0, 0
        test_loss = 0.0
        for batch in tqdm(test_loader, desc="Testing"):
            x, y = batch
            x, y = x.to(device), y.to(device)
            
            y_hat = model(x)
            loss = criterion(y_hat, y)
            test_loss += loss.detach().cpu().item() / len(test_loader)
            
            correct += torch.sum(torch.argmax(y_hat, dim=1) == y).detach().cpu().item()
            total += len(x)
        
        print(f"Test loss: {test_loss:.2f}")
        print(f"Test accuracy: {correct / total * 100:.2f}%")


# --- LANCEMENT ---
if __name__ == '__main__':
    main()

100%|██████████| 9.91M/9.91M [00:00<00:00, 29.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.10MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.52MB/s]


Using device:  cuda (Tesla T4)


Training:  20%|██        | 1/5 [05:48<23:13, 348.32s/it]              

Epoch 1/5 loss: 2.12



Training:  40%|████      | 2/5 [11:40<17:31, 350.52s/it]              

...

Epoch 4/5 loss: 1.70



Training: 100%|██████████| 5/5 [29:31<00:00, 354.31s/it]              


Epoch 5/5 loss: 1.68


Testing: 100%|██████████| 79/79 [00:33<00:00,  2.36it/s]

Test loss: 1.71
Test accuracy: 74.98%


### Synthèse de l'entraînement du ViT

| Epoch | Loss d'entraînement |
|---|---|
| 1 | 2.12 |
| 2 | 1.86 |
| 3 | 1.76 |
| 4 | 1.70 |
| 5 | 1.68 |

- La loss baisse régulièrement à chaque epoch (de 2.12 à 1.68), donc le modèle apprend.
- Le gain entre les epochs diminue (0.26 → 0.10 → 0.06 → 0.02), ce qui indique que la convergence est presque atteinte.
- La loss reste élevée (~1.7), ce qui suggère que le modèle a une **capacité limitée**.

#### Résultats finaux sur le jeu de test

| Métrique | Valeur |
|---|---|
| **Test loss** | 1.71 |
| **Test accuracy** | **74.98%** |
| **Temps d'entraînement** | 29 min 31 s (sur GPU T4) |
| **Nombre de paramètres** | ~10 000 |

#### Interprétation 

- **Accuracy de 74.98%** : le ViT classe correctement environ 7 500 images sur 10 000. C'est **nettement moins** que les 99% des modèles de la Partie 1.
- **Pourquoi cette limite** : avec `hidden_d=8` et seulement 2 blocs Transformer, le modèle a **~10 000 paramètres**, soit **10 fois moins** que le CNN (108 000). Il n'a pas la capacité d'apprendre finement les motifs des chiffres.
